# Setup

In [ ]:
SPARK_START_FROM_SCRATCH = True
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = ["10.15.20.1"]
SPARK_VPN_DOMAIN = "mavasbel.vpn.itam.mx"

SPARK_DOCKER_BASE = "spark:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JUPYTER_LAB_DOCKER_TAG = "spark-jupyter:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JOB_VENV_DOCKER_TAG = "spark-job-venv:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JOB_VENV_BUILD_DIR = "/opt/spark/venv-build"

SPARK_MASTER_NAME = "spark-master"
SPARK_MASTER_HOSTNAME = f"{SPARK_MASTER_NAME}.{SPARK_VPN_DOMAIN}"
# SPARK_MASTER_IP = "10.15.20.2"
SPARK_MASTER_WUBUI_PORT = 6080
SPARK_MASTER_PORT = 6077

SPARK_TOTAL_WORKERS = 3
SPARK_WORKER_NAMES = [f"spark-worker-{i+1}" for i in range(SPARK_TOTAL_WORKERS)]
SPARK_WORKER_HOSTNAMES = [
    f"{SPARK_WORKER_NAMES[i]}.{SPARK_VPN_DOMAIN}" for i in range(SPARK_TOTAL_WORKERS)
]
SPARK_WORKER_IPS = ["10.15.20.2"] * SPARK_TOTAL_WORKERS
SPARK_WORKER_WEBUI_PORTS = [6080 + (i + 1) for i in range(SPARK_TOTAL_WORKERS)]

SPARK_WORKDIR = "/opt/spark/work-dir"

JUPYTER_LAB_NAME = "spark-jupyter"
JUPYTER_LAB_HOSTNAME = f"{JUPYTER_LAB_NAME}.{SPARK_VPN_DOMAIN}"
# JUPYTER_LAB_IP = "10.15.20.2"
JUPYTER_LAB_PORT = 6888
JUPYTER_LAB_MONITOR_PORT = 4040
JUPYTER_LAB_TOKEN = ""

SPARK_SHARED_WORKSPACE = "shared-workspace"
SPARK_SHARED_WORKSPACE_DIR = f"/opt/spark/{SPARK_SHARED_WORKSPACE}"

In [2]:
HADOOP_NAMENODE_HOSTNAME = "namenode.mavasbel.vpn.itam.mx"
HADOOP_NAMENODE_IP = "10.15.20.2"
HADOOP_NAMENODE_PORT = 8020

In [3]:
import os
from pathlib import Path

SPARK_DATADIR = Path(os.path.join(os.path.abspath(Path.cwd()), "data"))
SPARK_DATADIR.mkdir(parents=True, exist_ok=True)

In [4]:
!pip install faker mimesis

##### Cleaning Spark context

In [5]:
import pyspark
from pyspark import SparkContext

sc = SparkContext.getOrCreate()
print(f"      Spark Version: {sc.version}")
print(f"    PySpark Version: {pyspark.__version__}")

try:
    sc.stop()
    print("🧹 Ghost SparkContext cleaned up.")
except Exception:
    print("✨ No existing SparkContext to clean.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 18:05:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


      Spark Version: 3.5.7
    PySpark Version: 3.5.7
🧹 Ghost SparkContext cleaned up.


# Spark session

In [6]:
import sys
from pyspark.sql import SparkSession
from datetime import datetime
from delta import configure_spark_with_delta_pip

SparkSession.builder.getOrCreate().stop()

builder = (
    SparkSession.builder.master(f"spark://{SPARK_MASTER_HOSTNAME}:{SPARK_MASTER_PORT}")
    .appName(f"SparkLab_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S.%f')}")
    .config("spark.archives", f"{SPARK_WORKDIR}/spark_job_env.tar.gz#environment")
    .config("spark.driver.host", f"{JUPYTER_LAB_HOSTNAME}")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "512m")
    .config("spark.executorEnv.PYSPARK_PYTHON", "./environment/bin/python3")
    .config("spark.executor.memory", "1G")
    .config(
        "spark.executorEnv.PYTHONPATH",
        f"./environment/lib/python{'.'.join(str(n) for n in sys.version_info[:2])}/site-packages",
    )
    .config("spark.sql.catalog.local.warehouse", os.path.abspath("iceberg-warehouse"))
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .enableHiveSupport()
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Scala Version: {spark._jvm.scala.util.Properties.versionString()}")
print("✅ Spark Session is now active.")

Spark Version: 3.5.7
Scala Version: version 2.12.18
✅ Spark Session is now active.


# Data generation

In [7]:
total_rows = 10000
partitions = 10


def batch_generator(ids):
    import socket
    import random
    from faker import Faker

    node_name = socket.gethostname()
    faker = Faker()
    for _ in ids:
        yield (
            faker.uuid4(),
            node_name,
            faker.date_time(),
            faker.first_name(),
            faker.last_name(),
            faker.email(),
            faker.basic_phone_number(),
            random.random() * 1000.0,
        )

df_column_names = [
    "id",
    "worker",
    "timestamp",
    "first_name",
    "last_name",
    "email",
    "phone",
    "amount",
]
df_column_types = spark.createDataFrame(
    list(batch_generator(range(1))), schema=df_column_names
).schema
print(f"✅ batch_generator schema: {df_column_types}")

✅ batch_generator schema: StructType([StructField('id', StringType(), True), StructField('worker', StringType(), True), StructField('timestamp', TimestampType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone', StringType(), True), StructField('amount', DoubleType(), True)])


In [8]:
from pyspark.sql import functions as F
from IPython.display import Markdown, display

df = spark.createDataFrame(
    list(batch_generator(range(total_rows))), df_column_names
).repartition(partitions)
df.write.mode("overwrite").csv(f"{SPARK_DATADIR}/faker.csv")
print(f"✅ Created {SPARK_DATADIR}/faker.csv")

partition_stats = (
    df.withColumn("partition_id", F.spark_partition_id())
    .groupBy("worker", "partition_id")
    .count()
    .orderBy("worker", "partition_id")
)
# partition_stats.show()
display(partition_stats.toPandas())


✅ Created /opt/spark/work-dir/data/faker.csv


,worker,partition_id,count
0,spark-jupyter.mavasbel.vpn.itam.mx,0,1002
1,spark-jupyter.mavasbel.vpn.itam.mx,1,1002
2,spark-jupyter.mavasbel.vpn.itam.mx,2,999
3,spark-jupyter.mavasbel.vpn.itam.mx,3,998
4,spark-jupyter.mavasbel.vpn.itam.mx,4,999
5,spark-jupyter.mavasbel.vpn.itam.mx,5,999
6,spark-jupyter.mavasbel.vpn.itam.mx,6,1000
7,spark-jupyter.mavasbel.vpn.itam.mx,7,999
8,spark-jupyter.mavasbel.vpn.itam.mx,8,1000
9,spark-jupyter.mavasbel.vpn.itam.mx,9,1002


In [9]:
from IPython.display import Markdown, display

rdd = spark.sparkContext.parallelize(range(total_rows), partitions).mapPartitions(
    batch_generator
)
df = rdd.toDF(df_column_names)
df.write.mode("overwrite").parquet(f"{SPARK_DATADIR}/faker.parquet")
print(f"✅ Created {SPARK_DATADIR}/faker.parquet")

partition_stats = (
    df.withColumn("partition_id", F.spark_partition_id())
    .groupBy("worker", "partition_id")
    .count()
    .orderBy("worker", "partition_id")
)
# partition_stats.show()
display(partition_stats.toPandas())

✅ Created /opt/spark/work-dir/data/faker.parquet


,worker,partition_id,count
0,spark-worker-1.mavasbel.vpn.itam.mx,2,1000
1,spark-worker-1.mavasbel.vpn.itam.mx,5,1000
2,spark-worker-1.mavasbel.vpn.itam.mx,6,1000
3,spark-worker-1.mavasbel.vpn.itam.mx,7,1000
4,spark-worker-2.mavasbel.vpn.itam.mx,0,1000
5,spark-worker-2.mavasbel.vpn.itam.mx,3,1000
6,spark-worker-2.mavasbel.vpn.itam.mx,9,1000
7,spark-worker-3.mavasbel.vpn.itam.mx,1,1000
8,spark-worker-3.mavasbel.vpn.itam.mx,4,1000
9,spark-worker-3.mavasbel.vpn.itam.mx,8,1000


In [10]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(df_column_types)
def generate_batch_vectorized(batch_ser: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(list(batch_generator(batch_ser)))


df: DataFrame = (
    spark.range(total_rows, numPartitions=partitions)
    .withColumn("data", generate_batch_vectorized("id"))
    .select("data.*")
)
# df.write.mode("overwrite").parquet(f"{SPARK_DATADIR}/faker_vectorized.parquet")
# df.coalesce(1).write.mode("overwrite").parquet(f"{SPARK_DATADIR}/faker_vectorized.parquet")
# pdf = df.toPandas()
# pdf.to_parquet(f"{SPARK_DATADIR}/faker_vectorized.parquet", index=False)
df.write.mode("overwrite").parquet(f"hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/faker_vectorized.parquet")
print(f"✅ Created hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/faker_vectorized.parquet")


partition_stats = (
    df.withColumn("partition_id", F.spark_partition_id())
    .groupBy("worker", "partition_id")
    .count()
    .orderBy("worker", "partition_id")
)
# partition_stats.show()
display(partition_stats.toPandas())

✅ Created hdfs://namenode.mavasbel.vpn.itam.mx:8020/tmp/faker_vectorized.parquet


,worker,partition_id,count
0,spark-worker-1.mavasbel.vpn.itam.mx,2,1000
1,spark-worker-1.mavasbel.vpn.itam.mx,5,1000
2,spark-worker-1.mavasbel.vpn.itam.mx,9,1000
3,spark-worker-2.mavasbel.vpn.itam.mx,1,1000
4,spark-worker-2.mavasbel.vpn.itam.mx,4,1000
5,spark-worker-2.mavasbel.vpn.itam.mx,6,1000
6,spark-worker-3.mavasbel.vpn.itam.mx,0,1000
7,spark-worker-3.mavasbel.vpn.itam.mx,3,1000
8,spark-worker-3.mavasbel.vpn.itam.mx,7,1000
9,spark-worker-3.mavasbel.vpn.itam.mx,8,1000


In [11]:
from IPython.display import Markdown, display
from pyspark.sql import functions as F

# Read it back and check the schema/count

# df_verify = spark.read.parquet(f"{SPARK_DATADIR}/faker_vectorized.parquet").repartition(partitions)
# pdf_verify = pd.read_parquet(f"{SPARK_DATADIR}/faker_vectorized.parquet")
# df_verify = spark.createDataFrame(pdf_verify).repartition(partitions)
df_verify = spark.read.parquet(f"hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/faker_vectorized.parquet").repartition(partitions)
print(f"Generated rows: {df_verify.count()}")

print("\nFirst 10 by timestamp desc:")
# df_verify.sort(F.col("timestamp").desc()).show(10)
display(df_verify.sort(F.col("timestamp").desc()).toPandas())

print("\nFirst 10 by count(first_name) desc:")
# df_verify.groupBy("first_name").count().sort(F.col("count").desc()).show(10)
display(df_verify.groupBy("first_name").count().sort(F.col("count").desc()).toPandas())

Generated rows: 10000

First 10 by timestamp desc:


,id,worker,timestamp,first_name,last_name,email,phone,amount
0,887ff57d-e76b-476f-81c3-62a10272ca3c,spark-worker-1.mavasbel.vpn.itam.mx,2026-04-19 12:59:18.533809,Jacob,Chavez,enguyen@example.com,587-882-8530,861.077491
1,cf4906ff-c395-4987-a9cc-93fdde46e3bc,spark-worker-1.mavasbel.vpn.itam.mx,2026-04-17 23:16:13.314915,Arthur,Hensley,lauraholland@example.org,487-780-6244,853.624506
2,13002eab-f26f-49eb-9bb4-d908b5ed6c66,spark-worker-1.mavasbel.vpn.itam.mx,2026-04-17 04:33:56.745696,Sean,Miller,mjones@example.com,(736)203-3216,213.819371
3,2e2eac1a-1fec-437e-bf00-ad34815617bd,spark-worker-1.mavasbel.vpn.itam.mx,2026-04-16 04:39:15.282836,Jamie,Brown,amandaheath@example.com,(489)908-7705,353.302703
4,2c16c71d-0f81-43ab-96cd-cc4605a7be6d,spark-worker-1.mavasbel.vpn.itam.mx,2026-04-16 02:01:19.905790,Kristina,Burke,qroman@example.com,489-248-9384,725.235815
...,...,...,...,...,...,...,...,...
9995,ca4dd9dc-f04c-4035-b1eb-78ab35f2c61d,spark-worker-1.mavasbel.vpn.itam.mx,1970-01-04 08:40:07.016203,Megan,Tyler,mitchelljennifer@example.com,4615696407,166.479486
9996,8bb1c810-205c-48a0-9df7-0706f3b66fe1,spark-worker-2.mavasbel.vpn.itam.mx,1970-01-04 01:54:22.731281,Scott,Morris,ebony96@example.net,504-781-2678,958.901948
9997,2b9f3340-ae65-432a-a88c-47a621e48c02,spark-worker-3.mavasbel.vpn.itam.mx,1970-01-03 23:44:06.782682,Zachary,Allison,smithalexander@example.com,(890)662-0699,657.426588
9998,7d2e6ab8-d45c-460c-abdc-e03c0c145c68,spark-worker-3.mavasbel.vpn.itam.mx,1970-01-03 19:17:52.888879,Courtney,Coffey,ugood@example.org,9484725956,40.502477



First 10 by count(first_name) desc:


,first_name,count
0,Michael,208
1,David,172
2,John,153
3,Jennifer,139
4,Christopher,137
...,...,...
653,Daisy,1
654,Makayla,1
655,Duane,1
656,Edgar,1


In [12]:
from IPython.display import Markdown, display

df_verify.createOrReplaceTempView("df_verify")
df_sparkql = spark.sql("""
    SELECT 
        first_name, 
        SUM(amount) as total_amount,
        COUNT(*) as first_name_count
    FROM df_verify
    GROUP BY first_name
    ORDER BY first_name_count DESC
""")
display(df_sparkql.toPandas())

,first_name,total_amount,first_name_count
0,Michael,108277.887146,208
1,David,90444.661333,172
2,John,80172.924500,153
3,Jennifer,69653.807023,139
4,Christopher,67181.520517,137
...,...,...,...
653,Devon,622.149819,1
654,Cristian,560.764923,1
655,Tanner,765.971391,1
656,Gavin,67.996416,1


In [ ]:
from IPython.display import Markdown, display

df_sparkql = spark.sql(f"""
    SELECT 
        first_name, 
        SUM(amount) as total_amount,
        COUNT(*) as first_name_count
    FROM parquet.`hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/faker_vectorized.parquet`
    GROUP BY first_name
    ORDER BY first_name_count DESC
""")
display(df_sparkql.toPandas())

26/04/19 18:05:38 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/19 18:05:38 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
